In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/FinPay_input.csv')

print("Datasetin ölçüsü (sətir, sütun):", df.shape)
display(df.head())
df.info()
display(df.describe(include='all'))

: 

In [ ]:
missing_df = pd.DataFrame({
    'Bos dəyərlərin sayı': df.isnull().sum(),
    'Faiz (%)': (df.isnull().sum() / len(df)) * 100
})
print("--{--- Boş dəyərlərin sayı ---}--")
display(missing_df[missing_df['Bos dəyərlərin sayı'] > 0])

tekrarlar = df.duplicated().sum()
print(f"\nTekrar setirlerin sayi:{tekrarlar}")

In [ ]:
cat_unique = ['gender','country']

for col in cat_unique:
    print(f"\nSütun: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 40)

In [ ]:
# Klassifikasiya
print("=== TARGET 1: CHURN PAYLANMASI (%) ===")
print(df['churn'].value_counts(normalize=True) * 100)

print("\n" + "="*40 + "\n")

# regression
print("=== TARGET 2: NEXT_MONTH_SPENDING İSTATİSTİKLERİ ===")
print(df['next_month_spending'].describe())

Faza 1 – Data Understanding Yekun Müşahidələri:
1. Dataset Həcmi: Dataset 18,072 sətir və 24 sütundan ibarətdir.
2. Çatışmayan Dəyərlər (Missing Values): "gender" (128), "country" (108), "monthly_income" (325) daxil olmaqla 7 sütunda az miqdarda eksik dəyər mövcuddur.
3. Kategorik Dəyərlərdə Uyğunsuzluq: "gender" sütununda eyni anlayışı bildirən müxtəlif yazılış formaları ("Female", "female", "F", "Male", "male", "M") var və standartlaşdırılmalıdır.
4. Dublikatlar və Anomaliyalar: Datasetdə 72 ədəd dublikat sətir tapılmışdır. Həmçinin "age" sütununda maks dəyərin 120 olması kimi anomaliyalar təmizlənməlidir.
5. Hədəf Dəyişənlərinin (Targets) Paylanması: "churn" sütununda böyük sinif balanssızlığı mövcuddur (%89.3-ə %10.7). "next_month_spending" göstəricisində isə maks dəyərin (14,266) yüksək olması outlierlerin cox olduğunu göstərir.

Faza 2 - EDA


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 5))
sns.scatterplot(data = df, x = 'monthly_income', y = 'monthly_spending', alpha = 0.5 , color = 'teal')
plt.title('Ayliq gelir vs ayliq xercleme')
plt.xlabel('ayliq gelir')
plt.ylabel('ayliq xercleme')
plt.show()

In [ ]:
plt.figure(figsize = (13,5))

plt.subplot(1, 2, 1)
sns.scatterplot(data = df, x = 'monthly_spending', y = 'next_month_spending', alpha = 0.5, color = 'royalblue')
plt.title('Monthly Spending vs Next Month Spending')
plt.xlabel('Cari Ay Xərcləməsi')
plt.ylabel('Növbəti Ay Xərcləməsi')

plt.subplot(1, 2, 2)
sns.scatterplot(data = df, x = 'previous_month_spending', y = 'next_month_spending', alpha = 0.5, color = 'darkgreen')
plt.title('Previous Month Spending vs Next Month Spending')
plt.xlabel('Əvvəlki Ay Xərcləməsi')
plt.ylabel('Növbəti Ay Xərcləməsi')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize = (12, 5))
plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='churn', y = 'app_logins', palette='Set2')
plt.title('Mobil Tətbiqə Daxilolma Sayı vs Churn')
plt.xlabel('Churn (0 = Qalan, 1 = Tərk edən)')
plt.ylabel('Tətbiqə Daxilolma Sayı (App Logins)')

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='churn', y = 'days_since_last_login', palette='Set2')
plt.title('Son Daxilolmadan Keçən Gün Sayı vs Churn')
plt.xlabel('Churn (0 = Qalan, 1 = Tərk edən)')
plt.ylabel('Keçən Gün Sayı (Days Since Last Login)')

plt.tight_layout()
plt.show()

In [ ]:
# 4. Transaction Frequency ve Churn Relation
plt.figure(figsize = (9, 5))

sns.boxplot(data = df, x = 'churn', y = 'transactions_last_month', palette = 'Set2')
plt.title('Transactions last month Frequency vs Churn')
plt.xlabel('Churn (0 = Qalan, 1 = Tərk edən)')
plt.ylabel('Transactions Frequency (Kecen ayki operation sayi)')

plt.tight_layout()
plt.show()


In [ ]:
print(df.columns.tolist())

In [ ]:
# 5. Support Tickets və Failed Transactions göstəricilərinin Churn ilə əlaqəsi
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.barplot(data = df, x = 'support_tickets', y = 'churn' , palette = 'magma')
plt.title('Support Tickets və Churn Nisbəti')
plt.xlabel('Support Tickets')
plt.ylabel('Churn Nisbəti')

plt.subplot(1, 2, 2)
sns.barplot(data = df, x = 'failed_transactions', y = 'churn', palette = 'magma')
plt.title('Failed Transactions və Churn Nisbəti')
plt.xlabel('Failed Transactions')
plt.ylabel('Churn Nisbəti')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig,axes = plt.subplots(1, 3, figsize = (18, 5))

sns.boxplot(data = df, x = 'churn', y = 'months_with_company' , palette = 'Set2', ax = axes[0])
axes[0].set_title('Tenure vs Churn')
axes[0].set_xlabel('Churn (0 = Qalan, 1 = Tərk edən)')
axes[0].set_ylabel('Tenure (Ay)')

sns.kdeplot(data = df, x = 'credit_score', palette = 'Set1', ax = axes[1])
axes[1].set_title('Credit Score Paylaşımı')
axes[1].set_xlabel('Credit Score')
axes[1].set_ylabel('Agirlik (Density)')

sns.scatterplot(data = df, x = 'transactions_last_month', y = 'account_balance', alpha = 0.6, palette = 'coolwarm', ax = axes[2])
axes[2].set_title('İşlem Sayı ve Balans')
axes[2].set_xlabel('Kecen Ayki İşlem Sayısı')
axes[2].set_ylabel('Balans (Balance)')

plt.tight_layout()
plt.show()

In [ ]:
df.columns

1) Dəstək Müraciətləri və Uğursuz Əməliyyatlar (Support Tickets & Failed Transactions):

Nəticə: Dəstək müraciətlərinin sayısı 3-dən yuxarı olan və 2-dən çox uğursuz əməliyyat yaşayan müştərilərdə Churn (tətbiqi tərk etmə) riski 30%-dən çox artır.

Biznes Mənası: Texniki xətalar və keyfiyyətsiz dəstək xidməti müştəri itkisinin əsas səbəbidir. Payment Gateway stabilləşdirilməli və dəstək xidməti sürətləndirilməlidir.

2) Müştəri Yaşı / İstifadə Müddəti (Months with Company):

Nəticə: Tətbiqdən ayrılan müştərilərin böyük hissəsi ilk 6-12 ay ərzində churn edənlərdir.

Biznes Mənası: Yeni qoşulan müştəriləri sistemdə saxlamaq üçün ilk aylarda xüsusi "Onboarding" bonusları və həvəsləndirici xidmətlər təklif edilməlidir.

3) Maliyyə Davranışı və Passiv Müştərilər (Account Balance vs Transactions):

Nəticə: Yüksək balansı olan, lakin ay ərzində çox az əməliyyat aparan passiv müştəri seqmenti mövcuddur.

Biznes Mənası: Bu müştərilərin aktivliyini artırmaq üçün xüsusi investisiya, cashback və ya depozit təklifləri göndərilməlidir.

FAZA 3 


DATA CLEANİNG

In [ ]:
#Dublikatların silinməsi
df_clean = df.drop_duplicates().reset_index(drop=True)
#Kateqoriya uyğunsuzluqlarının standartlaşdırılması
df_clean['gender'] = df_clean['gender'].replace({
    'female': 'Female', 'F': 'Female',
    'male': 'Male', 'M': 'Male'
})
df_clean['country'] = df_clean['country'].replace({
    'germany': 'Germany', 'DE': 'Germany',
    'U.K.': 'United Kingdom', 'UK': 'United Kingdom', 'United kingdom': 'United Kingdom'
})
#Biznes məntiqinə zidd anomaliyaların təmizlənməsi
df_clean.loc[df_clean['monthly_income'] < 0, 'monthly_income'] = np.nan
df_clean.loc[df_clean['days_since_last_login'] < 0, 'days_since_last_login'] = np.nan

#Missing value-lərin doldurulması
# Rəqəmsal sütunlar üçün median, kateqoriyal sütunlar üçün mode
for col in ['monthly_income', 'account_balance', 'credit_score', 'avg_transaction_value', 'app_logins']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in ['gender', 'country']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Təmizlənmədən sonra sətir və sütun sayi:", df_clean.shape)

FAZA 4

In [ ]:
#Debt-to-Income Ratio
df_clean['debt_to_income'] = df_clean['loan_amount'] / (df_clean['monthly_income'] + 1e-5)

#Transaction Intensity per tenure
df_clean['transaction_intensity'] = df_clean['transactions_last_month'] / (df_clean['months_with_company'] + 1e-5)

#Xercleme dinamikasi
df_clean['spending_momentum'] = df_clean['monthly_spending'] - df_clean['previous_month_spending']

#Engagement Score
df_clean['engagement_score'] = df_clean['app_logins'] / (df_clean['days_since_last_login'] + 1)

df_clean = df_clean.fillna(0)

Faza 5

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# X and y
X = df_clean.drop(columns=['customer_id', 'next_month_spending', 'churn'])
y_reg = df_clean['next_month_spending']

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

#Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

#Model Pipeline
reg_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

#Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

reg_pipeline.fit(X_train, y_train)
y_pred = reg_pipeline.predict(X_test)

#Metrics
print("--- Regression Consequences ---")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

FAZA 6

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_clf = df_clean['churn']

# Train / Test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1))
])

clf_pipeline.fit(X_train_c, y_train_c)
y_pred_c = clf_pipeline.predict(X_test_c)
y_prob_c = clf_pipeline.predict_proba(X_test_c)[:, 1]

print("--- Classification Hesabati ---")
print(classification_report(y_test_c, y_pred_c))
print(f"ROC-AUC Score: {roc_auc_score(y_test_c, y_prob_c):.4f}")